In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit
DATA_PATH = "data"
df_all = pd.read_csv(DATA_PATH) 
required_cols = {'Liver_regeneration_rate', 'Time_points', 'ID'}
missing = required_cols - set(df_all.columns)
if missing:
    raise KeyError(f"missing：{missing}")
strat_col = 'Time_points'
TEST_SIZE = 10          
split = StratifiedShuffleSplit(
    n_splits   = 1,
    test_size  = TEST_SIZE,
    random_state = 333
)
for train_idx, test_idx in split.split(df_all, df_all[strat_col]):
    train_new = df_all.iloc[train_idx].reset_index(drop=True)
    test_new  = df_all.iloc[test_idx].reset_index(drop=True)
print("training_set Time_points：")
print(train_new[strat_col].value_counts().sort_index(), "\n")

print("test_set Time_points：")
print(test_new[strat_col].value_counts().sort_index())
train_new.to_csv("new_train.csv",
                 index=False)
test_new.to_csv("new_test.csv",
                index=False)

In [ ]:
import numpy as np
from sklearn.base import clone
def oof_predict(estimator, X, y, cv):
    n   = len(y)
    oof = np.zeros(n)
    cnt = np.zeros(n)       
    for tr_idx, te_idx in cv.split(X, y):
        mdl = clone(estimator)
        mdl.fit(X.iloc[tr_idx], y[tr_idx])
        oof[te_idx] += mdl.predict(X.iloc[te_idx])
        cnt[te_idx] += 1
    oof /= cnt
    return oof

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import BayesianRidge
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import LeaveOneOut, GridSearchCV, KFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error
import warnings, re
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    message=re.escape("'squared' is deprecated in version 1.4 and will be removed in 1.6."),
    module=r"sklearn\.metrics\._regression"
)
RANDOM_SEED = 11
train_df = pd.read_csv("trainfeature5.csv")
test_df  = pd.read_csv("testfeature5.csv")
X_train_raw = train_df.drop(columns=["Liver_regeneration_rate"])
X_test_raw  = test_df.drop(columns=["Liver_regeneration_rate"])
y_train = train_df["Liver_regeneration_rate"].values
y_test  = test_df["Liver_regeneration_rate"].values
cat_cols = ["Time_points"]
num_cols = [c for c in X_train_raw.columns if c not in cat_cols]
num_pipe = Pipeline([
    ("log1p", FunctionTransformer(np.log1p, validate=False)),
    ("scale", StandardScaler())
])
cat_pipe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
preprocess = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])
param_grid = {
    "br__alpha_1": np.logspace(-6, -1, 4),
    "br__alpha_2": np.logspace(-6, -1, 4),
    "br__lambda_1": np.logspace(-6, -1, 4),
    "br__lambda_2": np.logspace(-6, -1, 4),
    "br__fit_intercept": [True, False]
}
model = Pipeline([
    ("prep", preprocess),
    ("br", BayesianRidge(tol=1e-6))
])
reg = TransformedTargetRegressor(
    regressor=model,
    transformer=StandardScaler()
)
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
grid = GridSearchCV(
    reg,
    param_grid=param_grid,
    cv=cv,
    scoring="r2",
    n_jobs=-1,
    verbose=1
)
grid.fit(X_train_raw, y_train)
print("\n=== optimum parameter ===")
print(grid.best_params_)
print(f"best cross-validation R² = {grid.best_score_:.3f}")
best_model = grid.best_estimator_
best_model.fit(X_train_raw, y_train)
y_pred_train = best_model.predict(X_train_raw)
y_pred_test  = best_model.predict(X_test_raw)
print("\n--- The final model performance ---")
def rmse(a,b): return root_mean_squared_error(a,b)
for tag, (yt, yp) in [("Train", (y_train, y_pred_train)), ("Test", (y_test, y_pred_test))]:
    print(f"{tag:<6}| R²={r2_score(yt, yp):.3f} "
          f"RMSE={rmse(yt, yp):.3f} MAE={mean_absolute_error(yt, yp):.3f}")
loo = LeaveOneOut()
y_pred_loo = np.zeros_like(y_train, dtype=float)
for tr, vl in loo.split(X_train_raw):
    mdl = grid.best_estimator_
    mdl.fit(X_train_raw.iloc[tr], y_train[tr])
    y_pred_loo[vl] = mdl.predict(X_train_raw.iloc[vl])
print("\n--- LOOCV result ---")
print(f"R² = {r2_score(y_train, y_pred_loo):.3f}")
print(f"RMSE = {rmse(y_train, y_pred_loo):.3f}")
print(f"MAE = {mean_absolute_error(y_train, y_pred_loo):.3f}")
def bootstrap_ci(metric_func, y_true, y_pred, B=1000, seed=42, alpha=0.05):
    rng = np.random.default_rng(seed)
    stats = [
        metric_func(y_true[idx := rng.integers(0, len(y_true), len(y_true))], y_pred[idx])
        for _ in range(B)
    ]
    return np.mean(stats), np.percentile(stats, [100*alpha/2, 100*(1-alpha/2)])
B = 1000
SEED = 42
print("\n=== Bootstrap 95% CI ===")
metrics = {
    "Train": (y_train, y_pred_train),
    "LOOCV": (y_train, y_pred_loo),
    "Test": (y_test, y_pred_test)
}
for tag, (yt, yp) in metrics.items():
    r2_mean, r2_ci = bootstrap_ci(r2_score, yt, yp, B=B, seed=SEED)
    rmse_mean, rmse_ci = bootstrap_ci(rmse, yt, yp, B=B, seed=SEED)
    mae_mean, mae_ci = bootstrap_ci(mean_absolute_error, yt, yp, B=B, seed=SEED)
    print(f"\n[{tag}]")
    print(f"• R²   = {r2_mean :.3f}  95% CI [{r2_ci[0]:.3f}, {r2_ci[1]:.3f}]")
    print(f"• RMSE = {rmse_mean:.3f} 95% CI [{rmse_ci[0]:.3f}, {rmse_ci[1]:.3f}]")
    print(f"• MAE  = {mae_mean :.3f} 95% CI [{mae_ci[0]:.3f}, {mae_ci[1]:.3f}]")
def bootstrap_table(y_true, y_pred, B=1000, seed=42):
    rng = np.random.default_rng(seed)
    res = [
        [
            r2_score(y_true[idx := rng.integers(0, len(y_true), len(y_true))], y_pred[idx]),
            mean_squared_error(y_true[idx], y_pred[idx], squared=False),
            mean_absolute_error(y_true[idx], y_pred[idx])
        ]
        for _ in range(B)
    ]
    return pd.DataFrame(res, columns=["R2", "RMSE", "MAE"])
train_df_ci = bootstrap_table(y_train, y_pred_loo, B=1000)
test_df_ci  = bootstrap_table(y_test, y_pred_test, B=1000)
train_df_ci.to_csv("train_bootstrap_results.csv", index=False)
test_df_ci.to_csv("test_bootstrap_results.csv", index=False)
print("\nThe result file has been exported.：train_bootstrap_results.csv, test_bootstrap_results.csv")

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings, sklearn
from pathlib import Path
from sklearn.model_selection  import GridSearchCV, LeaveOneOut
from sklearn.neighbors        import KNeighborsRegressor
from sklearn.pipeline         import Pipeline
from sklearn.compose          import ColumnTransformer
from sklearn.preprocessing    import (OneHotEncoder, StandardScaler, FunctionTransformer)
from sklearn.metrics          import (r2_score, mean_squared_error, mean_absolute_error)
warnings.filterwarnings("ignore")
RANDOM_SEED = 42
TRAIN_CSV = Path("trainfeature5.csv")
TEST_CSV  = Path("/testfeature5.csv")
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)
y_train = train_df["Liver_regeneration_rate"].values
y_test  = test_df ["Liver_regeneration_rate"].values
X_train = train_df.drop(columns=["Liver_regeneration_rate"])
X_test  = test_df .drop(columns=["Liver_regeneration_rate"])
cat_cols = ["Time_points"]
num_cols = X_train.columns.drop(cat_cols)
ohe = (OneHotEncoder(handle_unknown="ignore", sparse_output=False)
       if sklearn.__version__ >= "1.4"
       else OneHotEncoder(handle_unknown="ignore", sparse=False))
num_pipe = Pipeline([
    ("log",   FunctionTransformer(np.log1p, validate=False)),
    ("scale", StandardScaler())
])
preprocess = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", ohe,      cat_cols)
])
param_grid = {
    "knn__n_neighbors": [2, 3, 4, 5, 6, 8, 10, 12],
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2]
}
pipe = Pipeline([
    ("prep", preprocess),
    ("knn",  KNeighborsRegressor())
])
loo = LeaveOneOut()
grid = GridSearchCV(
    pipe,
    param_grid=param_grid,
    scoring="r2",
    cv=loo,
    n_jobs=-1
)
grid.fit(X_train, y_train)
best_pars = grid.best_params_
print("\n✅ The automatically selected optimal parameters：")
print(best_pars)
knn_best = grid.best_estimator_
y_pred_tr = knn_best.predict(X_train)
y_pred_te = knn_best.predict(X_test)
y_pred_loo = np.empty_like(y_train, dtype=float)
for tr, vl in loo.split(X_train):
    knn_cv = grid.best_estimator_
    knn_cv.fit(X_train.iloc[tr], y_train[tr])
    y_pred_loo[vl] = knn_cv.predict(X_train.iloc[vl])
try:
    from sklearn.metrics import root_mean_squared_error as rmse_fn
except ImportError:
    def rmse_fn(y_true, y_pred):
        return mean_squared_error(y_true, y_pred, squared=False)
def bootstrap_ci(y_true, y_pred, B=1000, seed=42):
    rng = np.random.default_rng(seed)
    r2s, rmses, maes = [], [], []
    n = len(y_true)
    for _ in range(B):
        idx = rng.integers(0, n, n)
        yt, yp = y_true[idx], y_pred[idx]
        r2s.append(r2_score(yt, yp))
        rmses.append(rmse_fn(yt, yp))
        maes.append(mean_absolute_error(yt, yp))
    pct = lambda arr: np.percentile(arr, [2.5, 97.5])
    return pct(r2s), pct(rmses), pct(maes)
met_train = (r2_score(y_train, y_pred_tr),
             rmse_fn(y_train, y_pred_tr),
             mean_absolute_error(y_train, y_pred_tr))
met_test  = (r2_score(y_test , y_pred_te),
             rmse_fn(y_test , y_pred_te),
             mean_absolute_error(y_test , y_pred_te))
met_loo   = (r2_score(y_train, y_pred_loo),
             rmse_fn(y_train, y_pred_loo),
             mean_absolute_error(y_train, y_pred_loo))
ci_test_r2, ci_test_rmse, ci_test_mae = bootstrap_ci(y_test, y_pred_te)
ci_loo_r2 , ci_loo_rmse , ci_loo_mae  = bootstrap_ci(y_train, y_pred_loo)
def pretty(line_name, point, ci_r2, ci_rmse, ci_mae):
    def _fmt(x): return f"{x:.3f}" if isinstance(x, (int,float,np.floating)) else str(x)
    print(f"{line_name:<6}| "
          f"R²={_fmt(point[0])} (95% CI {_fmt(ci_r2[0])}–{_fmt(ci_r2[1])}) | "
          f"RMSE={_fmt(point[1])} (95% CI {_fmt(ci_rmse[0])}–{_fmt(ci_rmse[1])}) | "
          f"MAE={_fmt(point[2])} (95% CI {_fmt(ci_mae[0])}–{_fmt(ci_mae[1])})")
print("\n===== KNN Final model performance =====")
pretty("Train", met_train, ["—","—"], ["—","—"], ["—","—"])
pretty("LOOCV", met_loo , ci_loo_r2 , ci_loo_rmse , ci_loo_mae)
pretty("Test ", met_test, ci_test_r2, ci_test_rmse, ci_test_mae)
def bootstrap_metrics_array(y_true, y_pred, B=1000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    r2_arr, rmse_arr, mae_arr = [], [], []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        yt_bs, yp_bs = y_true[idx], y_pred[idx]
        r2_arr.append(r2_score(yt_bs, yp_bs))
        rmse_arr.append(rmse_fn(yt_bs, yp_bs))
        mae_arr.append(mean_absolute_error(yt_bs, yp_bs))
    return np.array(r2_arr), np.array(rmse_arr), np.array(mae_arr)
r2_test_arr , rmse_test_arr , mae_test_arr  = bootstrap_metrics_array(y_test , y_pred_te)
r2_loo_arr  , rmse_loo_arr  , mae_loo_arr   = bootstrap_metrics_array(y_train, y_pred_loo)
r2_train_arr   = np.repeat(met_train[0], 1000)
rmse_train_arr = np.repeat(met_train[1], 1000)
mae_train_arr  = np.repeat(met_train[2], 1000)
df_knn = pd.DataFrame({
    "Metric": ["R2"]*3000 + ["RMSE"]*3000 + ["MAE"]*3000,
    "Group" : (["Train"]*1000 + ["Test"]*1000 + ["LOOCV"]*1000)*3,
    "Value" : np.concatenate([
        r2_train_arr,  r2_test_arr,  r2_loo_arr,
        rmse_train_arr,rmse_test_arr,rmse_loo_arr,
        mae_train_arr, mae_test_arr, mae_loo_arr
    ])
})
df_knn.to_csv("knn_metrics_for_origin.csv", index=False)
print("\n✅ Successfully saved the distribution of indicators as CSV：knn_metrics_for_origin.csv")

In [ ]:
import warnings, numpy as np, pandas as pd, sklearn
from pathlib import Path
from sklearn.linear_model    import Lasso
from sklearn.metrics         import (r2_score, mean_absolute_error,
                                     mean_squared_error)
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.preprocessing   import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.compose         import ColumnTransformer
from sklearn.pipeline        import Pipeline
warnings.filterwarnings("ignore")
RANDOM_SEED = 33
np.random.seed(RANDOM_SEED)
train_csv = Path("trainfeature5.csv")
test_csv  = Path("testfeature5.csv")
train = pd.read_csv(train_csv)
test  = pd.read_csv(test_csv)
y_train = train["Liver_regeneration_rate"].values
y_test  = test ["Liver_regeneration_rate"].values
X_train = train.drop(columns=["Liver_regeneration_rate"])
X_test  = test .drop(columns=["Liver_regeneration_rate"])
cat_cols = ["Time_points"]
num_cols = [c for c in X_train.columns if c not in cat_cols]
preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("log1p", FunctionTransformer(np.log1p, validate=False)),
        ("scale", StandardScaler())
    ]), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
])
best_pars = {"alpha": 0.001}
lasso_best = Pipeline([
    ("prep", preprocess),
    ("lasso", Lasso(alpha=best_pars["alpha"],
                    max_iter=10000,
                    random_state=RANDOM_SEED))
]).fit(X_train, y_train)
y_pred_tr = lasso_best.predict(X_train)
y_pred_te = lasso_best.predict(X_test)
loo = LeaveOneOut()
y_pred_loo = cross_val_predict(lasso_best, X_train, y_train, cv=loo, n_jobs=-1)
def rmse_fn(y_true, y_pred):
    return mean_squared_error(y_true, y_pred, squared=False)
def metrics(y_t, y_p):
    return (r2_score(y_t, y_p),
            rmse_fn(y_t, y_p),
            mean_absolute_error(y_t, y_p))
def bootstrap_ci(y_true, y_pred, B=1000, seed=42):
    rng = np.random.default_rng(seed)
    r2s, rmses, maes = [], [], []
    n = len(y_true)
    for _ in range(B):
        idx = rng.integers(0, n, n)
        yt, yp = y_true[idx], y_pred[idx]
        r2s  .append(r2_score(yt, yp))
        rmses.append(rmse_fn(yt, yp))
        maes .append(mean_absolute_error(yt, yp))
    pct = lambda arr: np.percentile(arr, [2.5, 97.5])
    return pct(r2s), pct(rmses), pct(maes)
met_train = metrics(y_train, y_pred_tr)
met_test  = metrics(y_test , y_pred_te)
met_loo   = metrics(y_train, y_pred_loo)
ci_test_r2, ci_test_rmse, ci_test_mae = bootstrap_ci(y_test , y_pred_te , B=1000, seed=42)
ci_loo_r2 , ci_loo_rmse , ci_loo_mae  = bootstrap_ci(y_train, y_pred_loo, B=1000, seed=42)
print("\n===== LASSO assess =====")
print(best_pars, "\n")
def _fmt(x):
    return f"{x:.3f}" if isinstance(x, (int, float, np.floating)) else str(x)
def pretty(line_name, point, ci_r2, ci_rmse, ci_mae):
    print(f"{line_name:<6}| "
          f"R²={_fmt(point[0]):>6}  (95% CI {_fmt(ci_r2[0])}–{_fmt(ci_r2[1])}) | "
          f"RMSE={_fmt(point[1]):>6}  (95% CI {_fmt(ci_rmse[0])}–{_fmt(ci_rmse[1])}) | "
          f"MAE={_fmt(point[2]):>6}  (95% CI {_fmt(ci_mae[0])}–{_fmt(ci_mae[1])})")
pretty("Train", met_train, ["—","—"], ["—","—"], ["—","—"])
pretty("LOOCV", met_loo  , ci_loo_r2 , ci_loo_rmse , ci_loo_mae)
pretty("Test ", met_test , ci_test_r2, ci_test_rmse, ci_test_mae)
def bootstrap_metrics_array(y_true, y_pred, B=1000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    r2_arr, rmse_arr, mae_arr = [], [], []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        yt_bs, yp_bs = y_true[idx], y_pred[idx]
        r2_arr.append(r2_score(yt_bs, yp_bs))
        rmse_arr.append(rmse_fn(yt_bs, yp_bs))
        mae_arr.append(mean_absolute_error(yt_bs, yp_bs))
    return np.array(r2_arr), np.array(rmse_arr), np.array(mae_arr)
r2_test_arr , rmse_test_arr , mae_test_arr  = bootstrap_metrics_array(y_test , y_pred_te , B=1000, seed=42)
r2_loo_arr  , rmse_loo_arr  , mae_loo_arr   = bootstrap_metrics_array(y_train, y_pred_loo, B=1000, seed=42)
r2_train, rmse_train, mae_train = met_train
r2_train_arr   = np.repeat(r2_train,   1000)
rmse_train_arr = np.repeat(rmse_train, 1000)
mae_train_arr  = np.repeat(mae_train,  1000)
df_lasso = pd.DataFrame({
    "Metric": ["R2"]   * 3000 + ["RMSE"] * 3000 + ["MAE"] * 3000,
    "Group" : (["Train"] * 1000 + ["Test"] * 1000 + ["LOOCV"] * 1000) * 3,
    "Value" : np.concatenate([
        r2_train_arr,  r2_test_arr,  r2_loo_arr,
        rmse_train_arr,rmse_test_arr,rmse_loo_arr,
        mae_train_arr, mae_test_arr, mae_loo_arr
    ])
})
df_lasso.to_csv("lasso_metrics_for_origin.csv", index=False)
print("\n✅ Successfully saved the distribution of indicators as CSV：lasso_metrics_for_origin.csv")

In [ ]:
import warnings, numpy as np, pandas as pd
from pathlib import Path
from sklearn.preprocessing   import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.compose         import ColumnTransformer
from sklearn.pipeline        import Pipeline
from sklearn.cross_decomposition import PLSRegression
from sklearn.compose         import TransformedTargetRegressor
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics         import (r2_score, mean_absolute_error,
                                     root_mean_squared_error)
warnings.filterwarnings("ignore")
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
TRAIN_CSV = Path("trainfeature5.csv")
TEST_CSV  = Path("testfeature5.csv")
train = pd.read_csv(TRAIN_CSV)
test  = pd.read_csv(TEST_CSV)
y_train = train["Liver_regeneration_rate"].values
y_test  = test ["Liver_regeneration_rate"].values
X_train = train.drop(columns=["Liver_regeneration_rate"])
X_test  = test .drop(columns=["Liver_regeneration_rate"])
cat_cols = ["Time_points"]
num_cols = [c for c in X_train.columns if c not in cat_cols]
preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("log1p", FunctionTransformer(np.log1p, validate=False)),
        ("z"    , StandardScaler())
    ]), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
])
n_comp = 3
pls_core = Pipeline([
    ("prep", preprocess),
    ("pls" , PLSRegression(n_components=n_comp, scale=True))
])
pls_model = TransformedTargetRegressor(regressor=pls_core,
                                       transformer=StandardScaler())
pls_model.fit(X_train, y_train)
y_pred_train = pls_model.predict(X_train)
y_pred_test  = pls_model.predict(X_test)
loo = LeaveOneOut()
y_pred_loocv = np.zeros_like(y_train, dtype=float)
for tr, vl in loo.split(X_train):
    mdl = TransformedTargetRegressor(regressor=pls_core,
                                     transformer=StandardScaler())
    mdl.fit(X_train.iloc[tr], y_train[tr])
    y_pred_loocv[vl] = mdl.predict(X_train.iloc[vl])
def metrics(y_t, y_p):
    return (r2_score(y_t, y_p),
            root_mean_squared_error(y_t, y_p),
            mean_absolute_error(y_t, y_p))
def bootstrap_ci(y_true, y_pred, B=1000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    r2_arr, rmse_arr, mae_arr = [], [], []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        r2, rmse, mae = metrics(y_true[idx], y_pred[idx])
        r2_arr.append(r2); rmse_arr.append(rmse); mae_arr.append(mae)
    p = lambda a: np.percentile(a, [2.5, 97.5])
    return p(r2_arr), p(rmse_arr), p(mae_arr)
def pretty(name, point, ci=None):
    r2, rmse, mae = point
    if ci is None:
        print(f"{name:<6}| R²={r2:6.3f}  RMSE={rmse:6.3f}  MAE={mae:6.3f}")
    else:
        (r2_l, r2_h), (rmse_l, rmse_h), (mae_l, mae_h) = ci
        print(f"{name:<6}| R²={r2:6.3f} (95% CI {r2_l:.3f}–{r2_h:.3f}) | "
              f"RMSE={rmse:6.3f} (95% CI {rmse_l:.3f}–{rmse_h:.3f}) | "
              f"MAE={mae:6.3f} (95% CI {mae_l:.3f}–{mae_h:.3f})")
pt_train = metrics(y_train, y_pred_train)
pt_test  = metrics(y_test , y_pred_test)
pt_loocv = metrics(y_train, y_pred_loocv)
ci_test  = bootstrap_ci(y_test , y_pred_test , B=1000, seed=88)
ci_loocv = bootstrap_ci(y_train, y_pred_loocv, B=1000, seed=90)
print(f"\n===== PLS (n_components = {n_comp}) =====")
pretty("Train", pt_train)
pretty("Test ", pt_test , ci_test)
pretty("LOOCV", pt_loocv, ci_loocv)
def bootstrap_metrics(y_true, y_pred, B=1000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    r2_arr, rmse_arr, mae_arr = [], [], []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        r2, rmse, mae = metrics(y_true[idx], y_pred[idx])
        r2_arr.append(r2); rmse_arr.append(rmse); mae_arr.append(mae)
    return pd.DataFrame({"R2": r2_arr, "RMSE": rmse_arr, "MAE": mae_arr})
pd.DataFrame([{
    "R2": pt_train[0],
    "RMSE": pt_train[1],
    "MAE": pt_train[2]
}]).to_csv("pls_train_point.csv", index=False)
bootstrap_metrics(y_test , y_pred_test , seed=88).to_csv("pls_test_bootstrap.csv", index=False)
bootstrap_metrics(y_train, y_pred_loocv, seed=90).to_csv("pls_loocv_bootstrap.csv", index=False)
print("\n✅ All the CSV files have been saved.：")
print("- pls_train_point.csv")
print("- pls_test_bootstrap.csv")
print("- pls_loocv_bootstrap.csv")

In [ ]:
import numpy as np, pandas as pd, warnings, copy, itertools
warnings.filterwarnings("ignore")
from pathlib import Path
from sklearn.preprocessing   import (OneHotEncoder, StandardScaler,
                                     FunctionTransformer)
from sklearn.compose         import ColumnTransformer
from sklearn.pipeline        import Pipeline
from sklearn.model_selection import RepeatedKFold
from sklearn.metrics         import (r2_score, mean_squared_error,
                                     mean_absolute_error)
from sklearn.neighbors      import KNeighborsRegressor
from sklearn.linear_model   import Lasso, Ridge, BayesianRidge  
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model   import LinearRegression
RANDOM_SEED = 42
TRAIN_CSV   = Path("trainfeature5.csv")
TEST_CSV    = Path("testfeature5.csv")
train = pd.read_csv(TRAIN_CSV)
test  = pd.read_csv(TEST_CSV)
y_train = train["Liver_regeneration_rate"].values
y_test  = test ["Liver_regeneration_rate"].values
metabs      = ["Com_2523_pos","Com_718_neg","Com_1720_pos",
               "Com_2201_pos","Com_768_neg"]
cat_cols    = ["Time_points"]     
num_cols    = metabs               
X_train_raw = train[cat_cols + metabs].copy()
X_test_raw  = test [cat_cols + metabs].copy()
preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("log1p", FunctionTransformer(np.log1p, validate=False)),
        ("z"    , StandardScaler())
    ]), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            cat_cols)
])
base_learners = {
    "br"   : Pipeline([("prep", preprocess),
                       ("mdl", BayesianRidge(alpha_1=0.01,
                         alpha_2=0.0001,
                         lambda_1=1.0,
                         lambda_2=1e-8,
                         fit_intercept=True))]),
    "knn"  : Pipeline([("prep", preprocess),
                       ("mdl" , KNeighborsRegressor(
                                    n_neighbors=4, p=1, weights="uniform"))]),
    "lasso": Pipeline([("prep", preprocess),
                       ("mdl" , Lasso(alpha=1e-3, max_iter=10_000,
                                      random_state=RANDOM_SEED))]),
    "pls"  : Pipeline([("prep", preprocess),
                       ("mdl" , PLSRegression(n_components=3, scale=True))])
}
rkf = RepeatedKFold(n_splits=5, n_repeats=10, random_state=RANDOM_SEED)
Z_train, Z_test = [], []
for name, pipe in base_learners.items():
    oof = np.zeros_like(y_train, dtype=float)
    for tr, vl in rkf.split(X_train_raw):
        mdl = copy.deepcopy(pipe)
        mdl.fit(X_train_raw.iloc[tr], y_train[tr])
        oof[vl] = mdl.predict(X_train_raw.iloc[vl])
    Z_train.append(oof)
    pipe.fit(X_train_raw, y_train)
    Z_test.append(pipe.predict(X_test_raw))
Z_train = np.vstack(Z_train).T       # shape = (n_train, 4)
Z_test  = np.vstack(Z_test ).T       # shape = (n_test , 4)
meta = LinearRegression().fit(Z_train, y_train)
stack_train_pred = meta.predict(Z_train)
stack_test_pred  = meta.predict(Z_test)
def met(y_t, y_p):
    return (r2_score(y_t, y_p),
            mean_squared_error(y_t, y_p, squared=False),
            mean_absolute_error(y_t, y_p))
rows = []
for name, pred_tr, pred_te in zip(base_learners.keys(), Z_train.T, Z_test.T):
    rows.append((f"{name}(tr)",) + met(y_train, pred_tr))
    rows.append((f"{name}(te)",) + met(y_test , pred_te ))
rows.append(("stack(tr)",) + met(y_train, stack_train_pred))
rows.append(("stack(te)",) + met(y_test , stack_test_pred ))
print("\nModel   |   R²     RMSE     MAE")
print("-"*41)
for r in rows:
    print(f"{r[0]:<9}| {r[1]:6.3f}  {r[2]:7.3f}  {r[3]:7.3f}")
import matplotlib.pyplot as plt
labels = [r[0].replace("(te)","") for r in rows if r[0].endswith("(te)")]
vals   = [r[1] for r in rows     if r[0].endswith("(te)")]
plt.figure(figsize=(7,4))
plt.bar(labels, vals, color="skyblue")
plt.ylabel("R² on test set"); plt.title("Stack vs. base models")
plt.xticks(rotation=20, ha="right")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
from itertools import combinations
from sklearn.linear_model import LinearRegression
names        = list(base_learners.keys())          
all_metrics  = []                                  
def _metric(y_t, y_p):
    return (r2_score(y_t, y_p),
            mean_squared_error(y_t, y_p, squared=False),
            mean_absolute_error(y_t, y_p))
for r in range(2, len(names)+1):                  
    for idxs in combinations(range(len(names)), r):
        combo_name = "+".join([names[i] for i in idxs])
        Z_tr_sub = Z_train[:, idxs]
        Z_te_sub = Z_test [:, idxs]
        meta = LinearRegression().fit(Z_tr_sub, y_train)
        y_tr_hat = meta.predict(Z_tr_sub)
        y_te_hat = meta.predict(Z_te_sub)
        all_metrics.append((
            combo_name, "train", *_metric(y_train, y_tr_hat)
        ))
        all_metrics.append((
            combo_name, "test" , *_metric(y_test , y_te_hat)
        ))
for i, nm in enumerate(names):
    all_metrics.append((nm, "train", *_metric(y_train, Z_train[:, i])))
    all_metrics.append((nm, "test" , *_metric(y_test , Z_test [:, i])))
df_out = (pd.DataFrame(all_metrics,
                       columns=["model","set","R2","RMSE","MAE"])
            .sort_values(["set","R2"], ascending=[False,False]))
print(df_out.to_string(index=False))

In [ ]:
import shap
import matplotlib.pyplot as plt
from pathlib import Path
save_dir = Path("./shap_plots")
save_dir.mkdir(exist_ok=True)
for i, name in enumerate(selected_learners, start=1):
    pipe = base_learners[name]
    pipe.fit(X_train_raw, y_train)
    X_trans = pipe.named_steps["prep"].transform(X_train_raw)
    explainer = shap.LinearExplainer(pipe.named_steps["mdl"], masker=shap.maskers.Independent(X_trans))
    shap_values = explainer(X_trans)
    shap_vals_2d = shap_values.values
    if len(shap_vals_2d.shape) == 3:
        shap_vals_2d = shap_vals_2d[:,:,0]
    metab_idx = np.arange(len(num_cols))  
    shap_vals_metab = shap_vals_2d[:, metab_idx]
    plt.figure()
    shap.summary_plot(shap_vals_metab, X_trans[:, metab_idx], feature_names=num_cols,
                      plot_type="bar", show=False)
    plt.title(f"SHAP summary for {name} (metabolites only)")
    plt.savefig(save_dir / f"SHAP{i}_metabs.png", dpi=1000, bbox_inches="tight")
    plt.close()